<a href="https://colab.research.google.com/github/uzaramen108/presentation/blob/260220/file/fenics_tutorial_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# setup

In [ ]:
# --------------------------------------------------
# 1️⃣ Mount Google Drive (optional, for cache)
# --------------------------------------------------
from google.colab import drive
import os

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
else:
    print("📦 Google Drive already mounted")

# --------------------------------------------------
# 2️⃣ Clone fenicsx-colab repository (idempotent)
# --------------------------------------------------
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/seoultechpse/fenicsx-colab.git"
ROOT = Path("/content")
REPO_DIR = ROOT / "fenicsx-colab"

def run(cmd):
    subprocess.run(cmd, check=True)

if not REPO_DIR.exists():
    print("📥 Cloning fenicsx-colab...")
    run(["git", "clone", REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / ".git").exists():
    raise RuntimeError("Directory exists but is not a git repository")
else:
    print("📦 Repository already exists — skipping clone")

# --------------------------------------------------
# 3️⃣ Run setup_fenicsx.py IN THIS KERNEL (CRITICAL)
# --------------------------------------------------
print("🚀 Running setup_fenicsx.py in current kernel")

# ⚙️ Configuration
USE_COMPLEX = False  # <--- Set True ONLY if you need complex PETSc
USE_CLEAN = True    # <--- Set True to remove existing environment

# Build options
opts = []
if USE_COMPLEX:
    opts.append("--complex")
if USE_CLEAN:
    opts.append("--clean")

opts_str = " ".join(opts) if opts else ""

get_ipython().run_line_magic(
    "run", f"{REPO_DIR / 'setup_fenicsx.py'} {opts_str}"
)

# --------------------------------------------------
# 4️⃣ Sanity check
# --------------------------------------------------
try:
    get_ipython().run_cell_magic('fenicsx', '--info -np 4', '')
except Exception as e:
    print("⚠️ %%fenicsx magic not found:", e)

Mounted at /content/drive
📥 Cloning fenicsx-colab...
🚀 Running setup_fenicsx.py in current kernel
🔧 FEniCSx Setup Configuration
PETSc type      : real
Clean install   : True

📦 Google Drive detected — using persistent cache

🔧 Installing FEniCSx environment...

🔍 Verifying PETSc type...
✅ Installed: Real PETSc (float64)

✨ Loading FEniCSx Jupyter magic... %%fenicsx registered

✅ FEniCSx setup complete!

Next steps:
  1. Run %%fenicsx --info to verify installation
  2. Use %%fenicsx in cells to run FEniCSx code
  3. Use -np N for parallel execution (e.g., %%fenicsx -np 4)

📌 Note: Real PETSc is installed
   - Recommended for most FEM problems
   - For complex problems, reinstall with --complex

🐍 Python          : 3.14.3
📦 dolfinx         : 0.10.0
💻 Platform        : Linux-6.6.105+-x86_64-with-glibc2.35
🧵 Running as root : True

🔎 fenicsx runtime info
-----------------------
Environment        : fenicsx
micromamba         : /content/micromamba/bin/micromamba
MPI implementation : OPENMPI

# 실행 파일

In [ ]:
%%fenicsx -np 4

import gmsh
import os
import numpy as np
import matplotlib.pyplot as plt

from mpi4py import MPI
from petsc4py import PETSc

from basix.ufl import element

from dolfinx.fem import (
    Constant,
    Function,
    functionspace,
    assemble_scalar,
    dirichletbc,
    extract_function_spaces,
    form,
    locate_dofs_topological,
    set_bc,
)
from dolfinx.fem.petsc import (
    apply_lifting,
    assemble_matrix,
    assemble_vector,
    create_vector,
    create_matrix,
    set_bc,
)
from dolfinx.geometry import bb_tree, compute_collisions_points, compute_colliding_cells
from dolfinx.io import VTXWriter, XDMFFile
from dolfinx.io import gmsh as gmshio
from ufl import (
    FacetNormal,
    Measure,
    TestFunction,
    TrialFunction,
    as_vector,
    div,
    dot,
    dx,
    inner,
    lhs,
    grad,
    nabla_grad,
    rhs,
)

# ============================================
# Mesh Generation with gmsh
# ============================================
gmsh.initialize()

L = 2.2
H = 0.41
c_x = c_y = 0.2
r = 0.05
gdim = 2
mesh_comm = MPI.COMM_WORLD
model_rank = 0

if mesh_comm.rank == model_rank:
    rectangle = gmsh.model.occ.addRectangle(0, 0, 0, L, H, tag=1)
    obstacle = gmsh.model.occ.addDisk(c_x, c_y, 0, r, r)

if mesh_comm.rank == model_rank:
    fluid = gmsh.model.occ.cut([(gdim, rectangle)], [(gdim, obstacle)])
    gmsh.model.occ.synchronize()

fluid_marker = 1
if mesh_comm.rank == model_rank:
    volumes = gmsh.model.getEntities(dim=gdim)
    assert len(volumes) == 1
    gmsh.model.addPhysicalGroup(volumes[0][0], [volumes[0][1]], fluid_marker)
    gmsh.model.setPhysicalName(volumes[0][0], fluid_marker, "Fluid")

inlet_marker, outlet_marker, wall_marker, obstacle_marker = 2, 3, 4, 5
inflow, outflow, walls, obstacle = [], [], [], []
if mesh_comm.rank == model_rank:
    boundaries = gmsh.model.getBoundary(volumes, oriented=False)
    for boundary in boundaries:
        center_of_mass = gmsh.model.occ.getCenterOfMass(boundary[0], boundary[1])
        if np.allclose(center_of_mass, [0, H / 2, 0]):
            inflow.append(boundary[1])
        elif np.allclose(center_of_mass, [L, H / 2, 0]):
            outflow.append(boundary[1])
        elif np.allclose(center_of_mass, [L / 2, H, 0]) or np.allclose(
            center_of_mass, [L / 2, 0, 0]
        ):
            walls.append(boundary[1])
        else:
            obstacle.append(boundary[1])
    gmsh.model.addPhysicalGroup(1, walls, wall_marker)
    gmsh.model.setPhysicalName(1, wall_marker, "Walls")
    gmsh.model.addPhysicalGroup(1, inflow, inlet_marker)
    gmsh.model.setPhysicalName(1, inlet_marker, "Inlet")
    gmsh.model.addPhysicalGroup(1, outflow, outlet_marker)
    gmsh.model.setPhysicalName(1, outlet_marker, "Outlet")
    gmsh.model.addPhysicalGroup(1, obstacle, obstacle_marker)
    gmsh.model.setPhysicalName(1, obstacle_marker, "Obstacle")

res_min = r / 3
if mesh_comm.rank == model_rank:
    distance_field = gmsh.model.mesh.field.add("Distance")
    gmsh.model.mesh.field.setNumbers(distance_field, "EdgesList", obstacle)
    threshold_field = gmsh.model.mesh.field.add("Threshold")
    gmsh.model.mesh.field.setNumber(threshold_field, "IField", distance_field)
    gmsh.model.mesh.field.setNumber(threshold_field, "LcMin", res_min)
    gmsh.model.mesh.field.setNumber(threshold_field, "LcMax", 0.25 * H)
    gmsh.model.mesh.field.setNumber(threshold_field, "DistMin", r)
    gmsh.model.mesh.field.setNumber(threshold_field, "DistMax", 2 * H)
    min_field = gmsh.model.mesh.field.add("Min")
    gmsh.model.mesh.field.setNumbers(min_field, "FieldsList", [threshold_field])
    gmsh.model.mesh.field.setAsBackgroundMesh(min_field)

if mesh_comm.rank == model_rank:
    gmsh.option.setNumber("Mesh.Algorithm", 8)
    gmsh.option.setNumber("Mesh.RecombinationAlgorithm", 2)
    gmsh.option.setNumber("Mesh.RecombineAll", 1)
    gmsh.option.setNumber("Mesh.SubdivisionAlgorithm", 1)
    gmsh.model.mesh.generate(gdim)
    gmsh.model.mesh.setOrder(2)
    gmsh.model.mesh.optimize("Netgen")

mesh_data = gmshio.model_to_mesh(gmsh.model, mesh_comm, model_rank, gdim=gdim)
mesh = mesh_data.mesh
assert mesh_data.facet_tags is not None
ft = mesh_data.facet_tags
ft.name = "Facet markers"

if mesh_comm.rank == 0:
    print(f"✅ Mesh created: {mesh.topology.index_map(gdim).size_global} cells")

# ============================================
# Physical Parameters
# ============================================
t = 0.0
T = 8.0  # Final time
dt = 0.001  # Time step size
num_steps = int(T / dt)
k = Constant(mesh, PETSc.ScalarType(dt))
mu = Constant(mesh, PETSc.ScalarType(2))  # Dynamic viscosity
rho = Constant(mesh, PETSc.ScalarType(1))  # Density

# ✅ 저장 간격 설정 (중요!)
save_interval = 50  # 100 스텝마다 저장 → 총 128개 파일

if mesh_comm.rank == 0:
    print(f"✅ Simulation parameters:")
    print(f"   T = {T}, dt = {dt}, num_steps = {num_steps}")
    print(f"   mu = {0.001}, rho = {1}")
    print(f"   Save interval: every {save_interval} steps ({num_steps//save_interval} files)")

# ============================================
# Function Spaces
# ============================================
v_cg2 = element("Lagrange", mesh.basix_cell(), 2, shape=(mesh.geometry.dim,))
s_cg1 = element("Lagrange", mesh.basix_cell(), 1)
V = functionspace(mesh, v_cg2)
Q = functionspace(mesh, s_cg1)

fdim = mesh.topology.dim - 1

# ============================================
# Boundary Conditions
# ============================================
class InletVelocity:
    def __init__(self, t):
        self.t = t

    def __call__(self, x):
        values = np.zeros((gdim, x.shape[1]), dtype=PETSc.ScalarType)
        values[0] = (
            4 * 1.5 * np.sin(self.t * np.pi / 8) * x[1] * (0.41 - x[1]) / (0.41**2)
        )
        return values


# Inlet
u_inlet = Function(V)
inlet_velocity = InletVelocity(t)
u_inlet.interpolate(inlet_velocity)
bcu_inflow = dirichletbc(
    u_inlet, locate_dofs_topological(V, fdim, ft.find(inlet_marker))
)
# Walls
u_nonslip = np.array((0,) * mesh.geometry.dim, dtype=PETSc.ScalarType)
bcu_walls = dirichletbc(
    u_nonslip, locate_dofs_topological(V, fdim, ft.find(wall_marker)), V
)
# Obstacle
bcu_obstacle = dirichletbc(
    u_nonslip, locate_dofs_topological(V, fdim, ft.find(obstacle_marker)), V
)
bcu = [bcu_inflow, bcu_obstacle, bcu_walls]
# Outlet
bcp_outlet = dirichletbc(
    PETSc.ScalarType(0), locate_dofs_topological(Q, fdim, ft.find(outlet_marker)), Q
)
bcp = [bcp_outlet]

if mesh_comm.rank == 0:
    print(f"✅ Boundary conditions applied")

# ============================================
# Variational Forms (IPCS)
# ============================================
u = TrialFunction(V)
v = TestFunction(V)
u_ = Function(V, name="u")
u_s = Function(V, name="u_tentative")
u_n = Function(V)
u_n1 = Function(V)
p = TrialFunction(Q)
q = TestFunction(Q)
p_ = Function(Q, name="p")
phi = Function(Q, name="phi")

# ✅ 초기 조건 설정 (중요!)
u_.x.array[:] = 0.0
u_n.x.array[:] = 0.0
u_n1.x.array[:] = 0.0
p_.x.array[:] = 0.0

f = Constant(mesh, PETSc.ScalarType((0, 0)))
F1 = rho / k * dot(u - u_n, v) * dx
F1 += inner(dot(1.5 * u_n - 0.5 * u_n1, 0.5 * nabla_grad(u + u_n)), v) * dx
F1 += 0.5 * mu * inner(grad(u + u_n), grad(v)) * dx - dot(p_, div(v)) * dx
F1 += dot(f, v) * dx
a1 = form(lhs(F1))
L1 = form(rhs(F1))
A1 = create_matrix(a1)
b1 = create_vector(extract_function_spaces(L1))

a2 = form(dot(grad(p), grad(q)) * dx)
L2 = form(-rho / k * dot(div(u_s), q) * dx)
A2 = assemble_matrix(a2, bcs=bcp)
A2.assemble()
b2 = create_vector(extract_function_spaces(L2))

a3 = form(rho * dot(u, v) * dx)
L3 = form(rho * dot(u_s, v) * dx - k * dot(nabla_grad(phi), v) * dx)
A3 = assemble_matrix(a3)
A3.assemble()
b3 = create_vector(extract_function_spaces(L3))

# ============================================
# Solvers
# ============================================
# Solver for step 1
solver1 = PETSc.KSP().create(mesh.comm)
solver1.setOperators(A1)
solver1.setType(PETSc.KSP.Type.BCGS)
pc1 = solver1.getPC()
pc1.setType(PETSc.PC.Type.JACOBI)

# Solver for step 2
solver2 = PETSc.KSP().create(mesh.comm)
solver2.setOperators(A2)
solver2.setType(PETSc.KSP.Type.MINRES)
pc2 = solver2.getPC()
pc2.setType(PETSc.PC.Type.HYPRE)
pc2.setHYPREType("boomeramg")

# Solver for step 3
solver3 = PETSc.KSP().create(mesh.comm)
solver3.setOperators(A3)
solver3.setType(PETSc.KSP.Type.CG)
pc3 = solver3.getPC()
pc3.setType(PETSc.PC.Type.SOR)

if mesh_comm.rank == 0:
    print(f"✅ Solvers configured")

# ============================================
# Force Coefficients
# ============================================
n = -FacetNormal(mesh)  # Normal pointing out of obstacle
dObs = Measure("ds", domain=mesh, subdomain_data=ft, subdomain_id=obstacle_marker)
u_t = inner(as_vector((n[1], -n[0])), u_)
drag = form(2 / 0.1 * (mu / rho * inner(grad(u_t), n) * n[1] - p_ * n[0]) * dObs)
lift = form(-2 / 0.1 * (mu / rho * inner(grad(u_t), n) * n[0] + p_ * n[1]) * dObs)

if mesh.comm.rank == 0:
    C_D = np.zeros(num_steps, dtype=PETSc.ScalarType)
    C_L = np.zeros(num_steps, dtype=PETSc.ScalarType)
    t_u = np.zeros(num_steps, dtype=np.float64)
    t_p = np.zeros(num_steps, dtype=np.float64)

# ============================================
# Pressure Difference Points
# ============================================
tree = bb_tree(mesh, mesh.geometry.dim)
points = np.array([[0.15, 0.2, 0], [0.25, 0.2, 0]])
cell_candidates = compute_collisions_points(tree, points)
colliding_cells = compute_colliding_cells(mesh, cell_candidates, points)
front_cells = colliding_cells.links(0)
back_cells = colliding_cells.links(1)
if mesh.comm.rank == 0:
    p_diff = np.zeros(num_steps, dtype=PETSc.ScalarType)

# ============================================
# Output Files
# ============================================
from pathlib import Path

folder = Path("results")
folder.mkdir(exist_ok=True, parents=True)

# ✅ VTX와 XDMF 둘 다 저장 (호환성)
vtx_u = VTXWriter(mesh.comm, folder / "dfg2D-3-u.bp", [u_], engine="BP4")
vtx_p = VTXWriter(mesh.comm, folder / "dfg2D-3-p.bp", [p_], engine="BP4")

# XDMF는 속도만 저장 (압력은 1차라서 2차 메시와 안맞음)
xdmf_u = XDMFFile(mesh.comm, folder / "dfg2D-3-u.xdmf", "w")
xdmf_u.write_mesh(mesh)

# 압력용 2차 함수 공간 생성 (XDMF 저장용)
s_cg2 = element("Lagrange", mesh.basix_cell(), 2)
Q2 = functionspace(mesh, s_cg2)
p_viz = Function(Q2, name="p")
xdmf_p = XDMFFile(mesh.comm, folder / "dfg2D-3-p.xdmf", "w")
xdmf_p.write_mesh(mesh)

# ✅ 초기 상태 저장
vtx_u.write(t)
vtx_p.write(t)
xdmf_u.write_function(u_, t)
# 압력을 2차로 보간하여 저장
p_viz.interpolate(p_)
xdmf_p.write_function(p_viz, t)

if mesh_comm.rank == 0:
    print(f"\n✅ Starting time loop...")

# ============================================
# Time Loop
# ============================================
for i in range(num_steps):
    # Update current time step
    t += dt
    # Update inlet velocity
    inlet_velocity.t = t
    u_inlet.interpolate(inlet_velocity)

    # Step 1: Tentative velocity step
    A1.zeroEntries()
    assemble_matrix(A1, a1, bcs=bcu)
    A1.assemble()
    with b1.localForm() as loc:
        loc.set(0)
    assemble_vector(b1, L1)
    apply_lifting(b1, [a1], [bcu])
    b1.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
    set_bc(b1, bcu)
    solver1.solve(b1, u_s.x.petsc_vec)
    u_s.x.scatter_forward()

    # Step 2: Pressure correction step
    with b2.localForm() as loc:
        loc.set(0)
    assemble_vector(b2, L2)
    apply_lifting(b2, [a2], [bcp])
    b2.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
    set_bc(b2, bcp)
    solver2.solve(b2, phi.x.petsc_vec)
    phi.x.scatter_forward()

    p_.x.petsc_vec.axpy(1, phi.x.petsc_vec)
    p_.x.scatter_forward()

    # Step 3: Velocity correction step
    with b3.localForm() as loc:
        loc.set(0)
    assemble_vector(b3, L3)
    b3.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
    solver3.solve(b3, u_.x.petsc_vec)
    u_.x.scatter_forward()

    # ✅ 저장 (save_interval마다만)
    if i % save_interval == 0:
        vtx_u.write(t)
        vtx_p.write(t)
        xdmf_u.write_function(u_, t)
        # 압력을 2차로 보간하여 저장
        p_viz.interpolate(p_)
        xdmf_p.write_function(p_viz, t)

    # Update variable with solution from this time step
    with (
        u_.x.petsc_vec.localForm() as loc_,
        u_n.x.petsc_vec.localForm() as loc_n,
        u_n1.x.petsc_vec.localForm() as loc_n1,
    ):
        loc_n.copy(loc_n1)
        loc_.copy(loc_n)

    # Compute physical quantities
    drag_coeff = mesh.comm.gather(assemble_scalar(drag), root=0)
    lift_coeff = mesh.comm.gather(assemble_scalar(lift), root=0)
    p_front = None
    if len(front_cells) > 0:
        p_front = p_.eval(points[0], front_cells[:1])
    p_front = mesh.comm.gather(p_front, root=0)
    p_back = None
    if len(back_cells) > 0:
        p_back = p_.eval(points[1], back_cells[:1])
    p_back = mesh.comm.gather(p_back, root=0)

    if mesh.comm.rank == 0:
        t_u[i] = t
        t_p[i] = t - dt / 2
        C_D[i] = sum(drag_coeff)
        C_L[i] = sum(lift_coeff)
        # Choose first pressure that is found from the different processors
        for pressure in p_front:
            if pressure is not None:
                p_diff[i] = pressure[0]
                break
        for pressure in p_back:
            if pressure is not None:
                p_diff[i] -= pressure[0]
                break

        # Progress output (every 500 steps)
        if (i + 1) % 500 == 0 or i == num_steps - 1:
            print(f"   Step {i+1:5d}/{num_steps}, t={t:.3f}, C_D={C_D[i]:.4f}, C_L={C_L[i]:.4f}")

vtx_u.close()
vtx_p.close()
xdmf_u.close()
xdmf_p.close()

if mesh_comm.rank == 0:
    print(f"\n✅ Simulation complete")

# ============================================
# Cleanup
# ============================================
A1.destroy()
A2.destroy()
A3.destroy()
b1.destroy()
b2.destroy()
b3.destroy()
solver1.destroy()
solver2.destroy()
solver3.destroy()

# ============================================
# Post-processing (Rank 0 only)
# ============================================
if mesh.comm.rank == 0:
    num_velocity_dofs = V.dofmap.index_map_bs * V.dofmap.index_map.size_global
    num_pressure_dofs = Q.dofmap.index_map_bs * Q.dofmap.index_map.size_global  # ✅ V→Q 버그 수정

    print(f"\n📊 Results:")
    print(f"   Total DOFs: {num_velocity_dofs + num_pressure_dofs}")
    print(f"   Velocity DOFs: {num_velocity_dofs}")
    print(f"   Pressure DOFs: {num_pressure_dofs}")
    print(f"   Max C_D: {np.max(C_D):.6f}")
    print(f"   Max C_L: {np.max(C_L):.6f}")
    print(f"   Max p_diff: {np.max(p_diff):.6f}")

    # Create figures directory
    if not os.path.exists("figures"):
        os.mkdir("figures")

    # ✅ Turek 벤치마크 데이터 로드 (파일 없을 경우 FEniCSx 단독 플롯으로 fallback)
    turek_available = os.path.exists("bdforces_lv4") and os.path.exists("pointvalues_lv4")
    if turek_available:
        turek   = np.loadtxt("bdforces_lv4")
        turek_p = np.loadtxt("pointvalues_lv4")
        print("✅ Turek benchmark data loaded — comparison plots will be generated")
    else:
        print("⚠️  'bdforces_lv4' / 'pointvalues_lv4' not found — plotting FEniCSx results only")

    # ── Drag coefficient ──────────────────────────────────────────────
    fig = plt.figure(figsize=(25, 8))
    plt.plot(
        t_u, C_D,
        label=r"FEniCSx ({0:d} dofs)".format(num_velocity_dofs + num_pressure_dofs),
        linewidth=2,
    )
    if turek_available:
        plt.plot(
            turek[1:, 1], turek[1:, 3],
            marker="x", markevery=50, linestyle="", markersize=4,
            label="FEATFLOW (42016 dofs)",
        )
    plt.title("Drag coefficient")
    plt.xlabel("Time [s]")
    plt.ylabel("C_D")
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.savefig("figures/drag_comparison.png", dpi=150)
    print("   📈 Saved: figures/drag_comparison.png")
    plt.close()

    # ── Lift coefficient ──────────────────────────────────────────────
    fig = plt.figure(figsize=(25, 8))
    plt.plot(
        t_u, C_L,
        label=r"FEniCSx ({0:d} dofs)".format(num_velocity_dofs + num_pressure_dofs),
        linewidth=2,
    )
    if turek_available:
        plt.plot(
            turek[1:, 1], turek[1:, 4],
            marker="x", markevery=50, linestyle="", markersize=4,
            label="FEATFLOW (42016 dofs)",
        )
    plt.title("Lift coefficient")
    plt.xlabel("Time [s]")
    plt.ylabel("C_L")
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.savefig("figures/lift_comparison.png", dpi=150)
    print("   📈 Saved: figures/lift_comparison.png")
    plt.close()

    # ── Pressure difference ───────────────────────────────────────────
    fig = plt.figure(figsize=(25, 8))
    plt.plot(
        t_p, p_diff,
        label=r"FEniCSx ({0:d} dofs)".format(num_velocity_dofs + num_pressure_dofs),
        linewidth=2,
    )
    if turek_available:
        plt.plot(
            turek[1:, 1], turek_p[1:, 6] - turek_p[1:, -1],
            marker="x", markevery=50, linestyle="", markersize=4,
            label="FEATFLOW (42016 dofs)",
        )
    plt.title("Pressure difference")
    plt.xlabel("Time [s]")
    plt.ylabel("Δp [Pa]")
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.savefig("figures/pressure_comparison.png", dpi=150)
    print("   📈 Saved: figures/pressure_comparison.png")
    plt.close()

    print(f"\n✅ All plots saved to 'figures/' directory")
    print(f"✅ Simulation data saved to 'results/' directory")
    print(f"\n💡 Tip: Use ParaView to open:")
    print(f"   - results/dfg2D-3-u.xdmf (velocity)")
    print(f"   - results/dfg2D-3-p.xdmf (pressure)")

Info    : [  0%] Difference                                                                                  
Info    : [ 10%] Difference                                                                                  
Info    : [ 20%] Difference                                                                                  
Info    : [ 30%] Difference - Performing Face-Face intersection                                                                                
Info    : [ 70%] Difference - Performing intersection of shapes                                                                                
Info    : [ 80%] Difference - Making faces                                                                                
Info    : [ 90%] Difference - Adding holes                                                                                
                                                                                
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 